# Week 3, day 4 (morning) — Worksheet 10 SOLUTIONS: quality checks and validation

Executed in the lab image. Every quoted number is what it actually printed.

Question 10 is the point of the whole worksheet. The model answers five of slide
25's six questions cleanly, and the sixth needs a table that does not exist —
which is a finding, not a failure.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 10 — Quality checks and validation. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr, tx = load("enrollment"), load("transaction")
crs, prg, coh = load("course"), load("program"), load("cohort")
stu, cat, dtype = load("students"), load("category"), load("discount_type")


def build_model():
    """Worksheets 06-09 in one function: the finished star schema."""
    test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
    spine = enr[~enr.stu_id.isin(test_ids)].copy()

    def keymap(df, col, sk):
        out = df[[col]].drop_duplicates().sort_values(col).reset_index(drop=True)
        out[sk] = range(1, len(out) + 1)
        out = out.rename(columns={col: "src"})
        return pd.concat([pd.DataFrame([{"src": -1, sk: -1}]), out],
                         ignore_index=True)

    dupes = tx.drop(columns=["trans_id"]).duplicated()
    per = (tx[~dupes].groupby("enrl_id")
           .agg(tuition_amount=("full_price", "max"),
                amount_paid_to_date=("payment_amount", "sum"),
                dtid=("discount_type_id", "max")).reset_index())
    per = per.merge(dtype[["discount_type_id", "discount_amount"]],
                    left_on="dtid", right_on="discount_type_id", how="left")

    f = spine.merge(crs[["course_id", "program_id"]], on="course_id", how="left")
    f = f.merge(per, on="enrl_id", how="left")
    f["dtid"] = f.dtid.fillna(-1)
    for df, left, sk in [(crs, "course_id", "course_id_sk"),
                         (prg, "program_id", "program_id_sk"),
                         (coh, "cohort_id", "cohort_id_sk"),
                         (stu, "stu_id", "student_id_sk"),
                         (dtype, "discount_type_id", "promotion_id_sk")]:
        src_col = {"course_id_sk": "course_id", "program_id_sk": "program_id",
                   "cohort_id_sk": "cohort_id", "student_id_sk": "stu_id",
                   "promotion_id_sk": "discount_type_id"}[sk]
        f = f.merge(keymap(df, src_col, sk).rename(columns={"src": "_s"}),
                    left_on=left, right_on="_s", how="left").drop(columns=["_s"])
        f[sk] = f[sk].fillna(-1).astype(int)

    f["enrollment_date_id"] = pd.to_datetime(f.enrl_date).dt.strftime("%Y%m%d").astype(int)
    for c in ("tuition_amount", "discount_amount", "amount_paid_to_date"):
        f[c] = f[c].fillna(0.0)
    f["net_tuition_amount"] = f.tuition_amount - f.discount_amount
    f["enrollment_count"] = 1
    f["is_cancelled"] = (f.status == "cancelled").astype(int)
    f["is_paid_in_full"] = ((f.net_tuition_amount > 0)
                            & (f.amount_paid_to_date >= f.net_tuition_amount - 0.005)
                            ).astype(int)
    fact = f.rename(columns={"enrl_id": "enrollment_id",
                             "program_id_sk": "program_key",
                             "course_id_sk": "course_key",
                             "cohort_id_sk": "cohort_key",
                             "student_id_sk": "student_key",
                             "promotion_id_sk": "promotion_key"})
    return spine, fact[["enrollment_id", "program_key", "course_key",
                        "cohort_key", "student_key", "enrollment_date_id",
                        "promotion_key", "enrollment_count", "tuition_amount",
                        "discount_amount", "net_tuition_amount",
                        "amount_paid_to_date", "is_paid_in_full", "is_cancelled"]]


spine, fact = build_model()

# dim_course carries program_category, so slide 25's questions need one join.
dim_course = (crs[["course_id", "course_name", "hours"]]
              .merge(prg[["program_id", "program_name", "category_id"]],
                     left_on=crs.program_id, right_on="program_id", how="left")
              .merge(cat[["category_id", "category_name"]], on="category_id",
                     how="left"))
dim_course["category_name"] = dim_course["category_name"].fillna("Unknown")
dim_course = dim_course.sort_values("course_id").reset_index(drop=True)
dim_course.insert(0, "course_key", range(1, len(dim_course) + 1))

dim_date = pd.DataFrame({"full_date": pd.date_range(
    pd.to_datetime(enr.enrl_date).min(), pd.to_datetime(enr.enrl_date).max(),
    freq="D")})
dim_date["date_id"] = dim_date.full_date.dt.strftime("%Y%m%d").astype(int)
dim_date["month"] = dim_date.full_date.dt.to_period("M").astype(str)
dim_date["quarter"] = dim_date.full_date.dt.quarter
dim_date["year"] = dim_date.full_date.dt.year

print("fact_enrollment:", fact.shape)
print("dim_course:     ", dim_course.shape)
print("dim_date:       ", dim_date.shape)

PART A — slide 39's five checks

### Question 1

**Row count check.** Compare `fact_enrollment`'s row count with the expected count derived independently from the source, and print both plus the difference.
> **NOTE:** derive the expectation from the source, not from the fact table. A check that reads its own answer checks nothing.

In [ ]:
test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
expected = len(enr) - int(enr.stu_id.isin(test_ids).sum())

print("source enrollment rows:      ", len(enr))
print("  less TEST-student rows:    ", int(enr.stu_id.isin(test_ids).sum()))
print("expected fact rows:          ", expected)
print("actual fact rows:            ", len(fact))
print("difference:                  ", len(fact) - expected)
print()
print("PASS" if len(fact) == expected else "FAIL")

```
source enrollment rows:       2400
  less TEST-student rows:     18
expected fact rows:           2382
actual fact rows:             2382
difference:                   0

PASS
```

2,400 minus 18 is 2,382, and the fact table has 2,382 rows.

The important property of this check is where the expectation comes from. `2382`
is **computed from the source** — the raw enrollment count minus the rows rule R5
removes — not read off the fact table and compared with itself. A check that
derives its expected value from the thing it is checking always passes.

That is what makes it the one check that would have caught worksheet 07 question
10's inner-join bug, where a fact table with 2,219 rows was internally perfect:
unique grain, no nulls, every measure present. Nothing inside the table was wrong.
The table was simply not all there.

Two refinements worth making in production:

**Show the subtraction.** The middle line — *less TEST-student rows: 18* — turns a
pass/fail into an explanation. When this check fails at 3am, the useful output is
not "expected 2382, got 2375"; it is which term of the expectation moved.

**Check the deltas too.** Absolute row counts catch a load that broke. Comparing
against the *previous* run catches a source that broke: 2,382 today and 240
tomorrow passes no sanity test, and this check as written would still say PASS if
the source itself had shrunk.

### Question 2

**Null check.** Print the null count for every column of `fact_enrollment`, and separately assert that the six key columns and five measures are complete.

In [ ]:
nulls = fact.isna().sum()
print("columns with nulls:", int((nulls > 0).sum()), "of", len(fact.columns))
print()
print(nulls.rename("nulls").to_string())
print()
KEYS = ["program_key", "course_key", "cohort_key", "student_key",
        "enrollment_date_id", "promotion_key"]
MEASURES = ["enrollment_count", "tuition_amount", "discount_amount",
            "net_tuition_amount", "amount_paid_to_date"]
print("keys complete:    ", bool(fact[KEYS].notna().all().all()))
print("measures complete:", bool(fact[MEASURES].notna().all().all()))

```
columns with nulls: 0 of 14

enrollment_id          0
program_key            0
...
is_cancelled           0

keys complete:     True
measures complete: True
```

Fourteen columns, no nulls anywhere.

That did not happen by itself. Every zero in that list is a decision made
somewhere in worksheets 06 to 09:

- the six keys are complete because of the `Unknown` members — 7 orphan students
  and 1,167 no-promotion rows would otherwise be nulls
- `tuition_amount`, `discount_amount` and `amount_paid_to_date` are complete
  because of the `fillna(0.0)` after the left join, covering the 156 enrollments
  with no transaction
- `discount_amount` is complete because of rule R3, covering 1,216 rows

Remove any one of those and this check fails.

**Why a null in a fact table is worth this much effort:** it is a decision
deferred to every future query, and different queries will decide differently.
`SUM` skips nulls, arithmetic propagates them, `groupby` drops rows on a null key,
and an inner join deletes them. Four behaviours, none of them announced, all from
one absent value.

The split between `keys complete` and `measures complete` is worth keeping
separate in a real suite, because the two failures mean different things. A null
key is a **load defect** — a lookup that did not resolve. A null measure is
usually a **rule that did not run**. Same symptom, different person to wake up.

Note that this check does not verify the values are *right* — only that they are
present. Question 5's business rules do that, and 156 rows of `tuition_amount = 0`
pass this check comfortably while being, as worksheet 07 question 9 argued, not
actually true.

### Question 3

**Duplicate check.** Test three things: `enrollment_id` unique, no fully duplicated rows, and the grain from worksheet 02 — no repeat of `student_key + course_key + cohort_key`.
> **NOTE:** the third one is expected to find something. Worksheet 02 question 2 explains what.

In [ ]:
print("rows:                        ", len(fact))
print("distinct enrollment_id:      ", fact.enrollment_id.nunique())
print("enrollment_id duplicated:    ", int(fact.enrollment_id.duplicated().sum()))
print("fully duplicated rows:       ", int(fact.duplicated().sum()))
print()
grain = ["student_key", "course_key", "cohort_key"]
d = int(fact.duplicated(grain).sum())
print("student+course+cohort duplicated:", d)
print()
if d:
    ex = fact[fact.duplicated(grain, keep=False)].sort_values(grain)
    print("first four rows involved:")
    print(ex[["enrollment_id"] + grain + ["enrollment_date_id"]]
          .head(4).to_string(index=False))

```
rows:                         2382
distinct enrollment_id:       2382
enrollment_id duplicated:     0
fully duplicated rows:        0

student+course+cohort duplicated: 16

first four rows involved:
 enrollment_id  student_key  course_key  cohort_key  enrollment_date_id
        701146           34          14           3            20240307
        701580           34          14           3            20240224
        700345          130          10          13            20250420
        701070          130          10          13            20250411
```

Two checks pass and the third finds 16 — and **the 16 are not a defect.**

These are worksheet 02 question 2's re-enrollments, arriving intact at the far end
of the pipeline: distinct `enrollment_id`, distinct dates, the same student in the
same course and cohort twice. Nine worksheets later they are still here, still
distinguishable, still countable.

That is the correct outcome, and it is only correct because worksheet 02 question
4 **restated the grain** to match. Had the model kept slide 27's original wording
— *one row per student's enrollment in a specific course and cohort* — this check
would be a genuine failure, and the fix would have been to delete 16 real
enrollments.

Which makes this the most instructive check on the sheet: **its expected value is
not zero, and knowing that requires having done worksheet 02.** A check suite
inherited from someone else, with `expected = 0` here, would fail every night and
be silenced within a week.

Three separate things were tested, and they are not redundant:

**`enrollment_id` duplicated** — the primary key. Catches a double load.

**Fully duplicated rows** — every column identical. Catches a duplicate that
somehow got distinct ids, which is exactly the shape of the 40 duplicate
`transaction` rows from worksheet 07 question 6.

**The business grain** — the semantic key. Catches rows that are legitimately
distinct records of the same event.

Run all three. They fail for different reasons.

### Question 4

**Referential integrity check.** For each of the five surrogate keys, print how many fact rows reference a key that does not exist in the dimension, and how many sit on the `Unknown` member.

In [ ]:
valid = {
    "course_key": set(range(1, len(crs) + 1)) | {-1},
    "program_key": set(range(1, len(prg) + 1)) | {-1},
    "cohort_key": set(range(1, len(coh) + 1)) | {-1},
    "student_key": set(range(1, len(stu) + 1)) | {-1},
    "promotion_key": set(range(1, len(dtype) + 1)) | {-1},
}
print("%-15s %10s %10s" % ("KEY", "ORPHANS", "ON UNKNOWN"))
for col, ok in valid.items():
    orphans = int((~fact[col].isin(ok)).sum())
    unknown = int((fact[col] == -1).sum())
    print("%-15s %10d %10d" % (col, orphans, unknown))
print()
dates = set(dim_date.date_id)
print("%-15s %10d %10s" % ("date_id",
                           int((~fact.enrollment_date_id.isin(dates)).sum()), "n/a"))

```
KEY                ORPHANS ON UNKNOWN
course_key               0          0
program_key              0          0
cohort_key               0          0
student_key              0          7
promotion_key            0       1167

date_id                  0        n/a
```

**Zero orphans across every key** — every value in the fact table exists in its
dimension. That is the guarantee a foreign key is supposed to provide, and in a
warehouse it usually has to be checked rather than enforced: Snowflake accepts
`FOREIGN KEY` declarations and does not enforce them, and most columnar platforms
are the same.

The two columns say different things and both matter.

**Orphans are always a defect.** A fact row pointing at a dimension key that does
not exist means the load ran against a stale dimension, or the key lookup was
wrong. Expected value: 0, always, for every key.

**`ON UNKNOWN` is a monitored number, not a pass/fail.** 7 orphan students is a
known anomaly. 1,167 no-promotion rows is normal business. Neither is an error —
and either one moving is a signal.

That distinction is why the two columns are printed side by side. Collapsing them
into one "referential integrity: PASS" hides the interesting number. The check
that would actually catch a problem here is *"`student_key = -1` is 7, and was 7
yesterday"*.

Note the last line. `date_id` has no `Unknown` member, so its orphan count must be
zero and there is no fallback — a fact row dated outside `dim_date`'s range simply
has no valid key. That is the failure that arrives on its own schedule, when data
appears past the last date the dimension was generated for, and it is why date
dimensions are built years ahead rather than fitted to the data.

**Where this check belongs:** immediately after the fact load, before anything
downstream reads the table. An orphan key found in a dashboard is a broken
dashboard; found here it is a re-run.

### Question 5

**Business rule check.** Test slide 39's two rules — `discount_amount <= tuition_amount`, and `is_paid_in_full` recomputed from the measures — and print the violation count for each.

In [ ]:
v1 = int((fact.discount_amount > fact.tuition_amount).sum())
v2 = int((fact.net_tuition_amount
          != (fact.tuition_amount - fact.discount_amount)).sum())
recomputed = ((fact.net_tuition_amount > 0)
              & (fact.amount_paid_to_date >= fact.net_tuition_amount - 0.005)
              ).astype(int)
v3 = int((fact.is_paid_in_full != recomputed).sum())
v4 = int((fact.net_tuition_amount < 0).sum())

for name, n in [("discount_amount <= tuition_amount", v1),
                ("net = tuition - discount", v2),
                ("is_paid_in_full recomputes", v3),
                ("net_tuition_amount >= 0", v4)]:
    print("  %-36s %5d violations  %s" % (name, n, "PASS" if n == 0 else "FAIL"))
print()
print("enrollments with zero tuition (no transaction): %d, of which "
      "is_paid_in_full=1: %d"
      % (int((fact.tuition_amount == 0).sum()),
         int(fact.loc[fact.tuition_amount == 0, "is_paid_in_full"].sum())))

```
  discount_amount <= tuition_amount        0 violations  PASS
  net = tuition - discount                 0 violations  PASS
  is_paid_in_full recomputes               0 violations  PASS
  net_tuition_amount >= 0                  0 violations  PASS

enrollments with zero tuition (no transaction): 156, of which is_paid_in_full=1: 0
```

All four pass, and the last line is the one that had to be earned.

**156 enrollments have zero tuition, and none of them is marked paid in full.**
Worksheet 07 question 9 showed the naive version of this flag marking all 156 as
settled, because `0 >= 0` is true — inflating the headline full-payment rate by
2.44 percentage points and telling the collections team that 156 non-payers had
paid. The `net_tuition_amount > 0` guard is what fixed it, and this line is the
evidence that the guard is still in place.

The third check is the pattern worth stealing: **`is_paid_in_full` is recomputed
from the stored measures and compared with the stored flag.** It is a redundancy
check — the flag and the measures are two representations of the same fact, and
they must agree.

That catches the class of bug where a measure is corrected and a derived flag is
not. Fix `amount_paid_to_date` in one place, forget the flag, and the table now
contains a row that says it was paid in full alongside numbers that say it was
not. Nothing else in this suite notices; every column is present, every key
resolves, the row count is right.

The fourth check — `net_tuition_amount >= 0` — is the one currently passing with
the most room. Worksheet 07 question 5 measured the closest approach at 1,500, so
a promotion worth more than a course fee would trip it. Zero violations today is
worth less than knowing how close the margin is.

**Business rule checks are the only checks that encode what the business
actually means.** The other four in this suite would pass on a table full of
plausible nonsense.

### Question 6

Turn the five checks into one harness: a list of (name, expected, actual) tuples that prints a PASS/FAIL table and a final verdict.
> **NOTE:** a check suite that only prints numbers is a report. One that prints PASS/FAIL is a gate.

In [ ]:
test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
expected_rows = len(enr) - int(enr.stu_id.isin(test_ids).sum())
recomputed = ((fact.net_tuition_amount > 0)
              & (fact.amount_paid_to_date >= fact.net_tuition_amount - 0.005)
              ).astype(int)

CHECKS = [
    ("row count matches source", expected_rows, len(fact)),
    ("enrollment_id unique", 0, int(fact.enrollment_id.duplicated().sum())),
    ("no nulls in keys or measures", 0,
     int(fact.drop(columns=["is_cancelled"]).isna().sum().sum())),
    ("no fully duplicated rows", 0, int(fact.duplicated().sum())),
    ("orphan student keys", 7, int((fact.student_key == -1).sum())),
    ("discount <= tuition", 0,
     int((fact.discount_amount > fact.tuition_amount).sum())),
    ("is_paid_in_full recomputes", 0,
     int((fact.is_paid_in_full != recomputed).sum())),
]
print("%-32s %10s %10s %6s" % ("CHECK", "EXPECTED", "ACTUAL", ""))
failed = 0
for name, exp, act in CHECKS:
    ok = exp == act
    failed += not ok
    print("%-32s %10d %10d %6s" % (name, exp, act, "PASS" if ok else "FAIL"))
print()
print("%d of %d checks passed" % (len(CHECKS) - failed, len(CHECKS)))

```
CHECK                              EXPECTED     ACTUAL
row count matches source               2382       2382   PASS
enrollment_id unique                      0          0   PASS
no nulls in keys or measures              0          0   PASS
no fully duplicated rows                  0          0   PASS
orphan student keys                       7          7   PASS
discount <= tuition                       0          0   PASS
is_paid_in_full recomputes                0          0   PASS

7 of 7 checks passed
```

Seven checks, one table, one verdict. This is the artefact slide 41 means by
*"data quality checks and expected outcomes"*.

Three properties make it a gate rather than a report.

**Every check has an expected value written down.** Including `orphan student
keys = 7`, which is the one that proves the point. Seven is not zero and it is not
a failure — it is a known anomaly with an open question behind it, and encoding it
as *expected* means the check fails when it becomes 8. A suite that only asserts
zeros cannot express "this is wrong but known", so those cases get dropped from
the suite entirely, and then nobody notices when they change.

**It produces a single boolean.** `7 of 7` is what a pipeline can branch on: hold
the load, don't publish, page someone. A wall of numbers requires a human to read
it, and at 3am nobody does.

**The expectations come from outside the table.** 2,382 is derived from the
source; 7 is a recorded fact about known bad data. Nothing here compares the fact
table with itself.

Two things this suite is missing, and both are worth adding before trusting it:

**Trend checks.** Every expectation here is absolute. A source that silently
halves would still pass every one of them. Comparing row counts, sums and
`Unknown` counts against the previous run catches what absolutes cannot.

**A reconciliation of money.** Worksheet 09 question 9 compared
`SUM(amount_paid_to_date)` against the source and got a difference of 0.00. That
check is more sensitive than all seven of these to aggregation errors, and it
belongs in the suite.

PART B — slide 25's six business questions

### Question 7

**Q1 and Q2.** *How many students enrolled each day?* and *how is enrollment changing over time by course?* Answer both from the model, printing the three busiest days and enrollments by quarter for one course.

In [ ]:
by_day = fact.groupby("enrollment_date_id").enrollment_count.sum()
print("busiest enrollment days:")
print(by_day.sort_values(ascending=False).head(3).to_string())
print("days with enrollments:", len(by_day))
print()
j = (fact.merge(dim_date[["date_id", "quarter", "year"]],
                left_on="enrollment_date_id", right_on="date_id")
         .merge(dim_course[["course_key", "course_name"]], on="course_key"))
one = j[j.course_name == "Data Modeling and ETL Design"]
print("'Data Modeling and ETL Design' enrollments by quarter:")
print(one.groupby(["year", "quarter"]).enrollment_count.sum().to_string())

```
busiest enrollment days:
enrollment_date_id
20250706    14
20240803    14
20241207    12
days with enrollments: 650

'Data Modeling and ETL Design' enrollments by quarter:
year  quarter
2023  4           2
2024  1          15
      2          10
      3          13
      4          12
2025  1           9
      2           8
      3          20
```

Slide 25's first two questions, answered.

**Question 1** — *how many students enrolled each day?* — is
`SUM(enrollment_count) GROUP BY enrollment_date_id`. One table, one group-by, no
joins.

Compare that with worksheet 01. The same question against the source needed the
`enrollment` table, a date parse, and knowledge of which of twelve tables held the
event. And worksheet 02 question 6 showed a coarser grain making it **impossible**
— which is why the grain decision came first.

**Question 2** — *how is enrollment changing over time by course?* — needs two
joins, to `dim_date` for the quarter and `dim_course` for the name. Note that
`quarter` is a **column**, not an expression: no `EXTRACT`, no date arithmetic, no
two analysts disagreeing about fiscal quarters. That is what the date dimension is
for.

The answer itself is interesting. This course runs 10 to 15 enrollments a quarter
and then jumps to **20 in 2025 Q3** — the highest in its history, roughly double
the preceding quarter. Whether that is a trend, a marketing campaign, or an
artefact of the data ending on 2025-09-28 mid-quarter is exactly the follow-up
question a working model is supposed to provoke.

And 2023 Q4 shows 2 enrollments, because the data starts on 2023-11-24. **A first
and last period that are partial will always look like a collapse**, and it is the
most common misreading of a time series from a warehouse. `dim_date` makes that
diagnosable — you can see the range — but it does not make it obvious.

### Question 8

**Q3 and Q4.** *Which courses have the highest enrollment?* and *which course-cohort groups have the highest full payment rate?* Answer both, and apply worksheet 03 question 7's lesson to the second.
> **NOTE:** a rate without its denominator is not an answer. Slide 30 says to benchmark; do the minimum version of that.

In [ ]:
j = fact.merge(dim_course[["course_key", "course_name"]], on="course_key")
print("top 5 courses by enrollment:")
print(j.groupby("course_name").enrollment_count.sum()
       .sort_values(ascending=False).head(5).to_string())
print()
g = fact.groupby(["course_key", "cohort_key"]).agg(
    n=("enrollment_count", "sum"),
    paid=("is_paid_in_full", "sum"))
g["rate"] = g.paid / g.n
print("course-cohort groups:", len(g))
print("groups with 8+ enrollments:", int((g.n >= 8).sum()))
print()
print("top 5 full-payment rates, UNFILTERED:")
print(g.sort_values("rate", ascending=False)[["n", "rate"]].head(5).round(3).to_string())
print()
print("top 5 among groups with 8+ enrollments:")
print(g[g.n >= 8].sort_values("rate", ascending=False)[["n", "rate"]]
      .head(5).round(3).to_string())

```
top 5 courses by enrollment:
course_name
Streaming Fundamentals         120
Distributed Systems Primer     120
Feature Engineering            119
Capstone Project               112
Python for Data Engineering    112

course-cohort groups: 383
groups with 8+ enrollments: 115

top 5 full-payment rates, UNFILTERED:
                       n  rate
course_key cohort_key
20         10          6   1.0
21         9           4   1.0
           16          5   1.0
3          8           4   1.0
           7           9   1.0

top 5 among groups with 8+ enrollments:
                       n   rate
course_key cohort_key
3          7           9  1.000
16         9           9  0.889
10         7           8  0.875
22         1           8  0.875
14         11          8  0.875
```

**Question 3** is clean: five courses between 112 and 120 enrollments, and the top
two are tied at 120. Worth noting the spread — 120 against 112 is a 7% gap across
five courses, so "the most popular course" is not a robust claim. Same caution as
worksheet 03 question 4.

**Question 4 is where the model needs help.** The unfiltered ranking returns five
groups with a **perfect 100% full-payment rate**, and their denominators are 6, 4,
5, 4 and 9. Four of the five are groups where fewer than six people enrolled and
all of them happened to pay.

That is not a finding about payment behaviour. It is arithmetic on small numbers,
and it is worksheet 03 question 7 recurring — there the top discount-rate group
had **n = 1**.

Filtered to groups with 8 or more, the answer becomes usable: one group at 100%
over 9 enrollments, then 0.889 and three at 0.875. Real variation, real
denominators, and course 3 appears in both lists, which makes it worth looking at.

**115 of 383 groups clear the threshold**, so the filter discards 70% of the
groups — which is itself the finding. Course-cohort is a fine grain for *storing*
data and a thin one for *ranking* it. The honest presentations are either the
filtered ranking with `n` shown, or a rollup to course level where every
denominator is 70+.

Slide 30's answer to question 4 is *"calculate SUM(is_paid_in_full) /
SUM(enrollment_count) by dim_course and dim_cohort"*, which is exactly the
computation here — and it says nothing about denominators. **The model supports
the question; it does not stop you answering it badly.**

### Question 9

**Q5 and Q6.** *Which course-cohort groups have the highest discount rates?* and *which may be over-discounted or underperforming?* Compute the discount rate as a ratio of sums, then find groups that are above the course benchmark on discount and below it on full payment.

In [ ]:
g = fact.groupby(["course_key", "cohort_key"]).agg(
    n=("enrollment_count", "sum"),
    disc=("discount_amount", "sum"),
    tuition=("tuition_amount", "sum"),
    paid=("is_paid_in_full", "sum")).reset_index()
g["discount_rate"] = g.disc / g.tuition
g["payment_rate"] = g.paid / g.n
g = g[g.n >= 8]

bench = g.groupby("course_key").agg(
    course_discount=("discount_rate", "mean"),
    course_payment=("payment_rate", "mean"))
g = g.merge(bench, on="course_key")
flagged = g[(g.discount_rate > g.course_discount)
            & (g.payment_rate < g.course_payment)]

print("groups with 8+ enrollments:", len(g))
print("above their course's discount rate AND below its payment rate:",
      len(flagged))
print()
print(flagged.sort_values("discount_rate", ascending=False)
      [["course_key", "cohort_key", "n", "discount_rate", "payment_rate",
        "course_discount", "course_payment"]].head(5).round(3).to_string(index=False))

```
groups with 8+ enrollments: 115
above their course's discount rate AND below its payment rate: 21

 course_key  cohort_key  n  discount_rate  payment_rate  course_discount  course_payment
         18          14 10          0.196         0.500            0.138           0.560
          1          15 10          0.192         0.500            0.096           0.545
         18           2  9          0.188         0.556            0.138           0.560
          3           3  8          0.162         0.500            0.097           0.665
          8           4  9          0.141         0.444            0.099           0.501
```

**Question 5** — highest discount rates — is `SUM(discount_amount) /
SUM(tuition_amount)`, the ratio of sums. Worksheet 03 question 6 established why:
the mean of per-enrollment rates answers a different question, and this one is
about money.

**Question 6** is the interesting one, and it is the only question on slide 25 that
does not have a single obvious formula. Slide 30's guidance is *"compare enrollment
volume, full payment rate, and discount rate by dim_course and dim_cohort against
course-level benchmarks"* — three measures, compared against a benchmark, and the
benchmark has to be constructed.

The construction here is the simplest defensible one: a group is flagged if it
discounted **more** than its course's average *and* collected **less**. That is
the definition of paying more to get worse customers.

**21 of 115 groups — 18%** — meet both conditions.

Read the top row. Course 18, cohort 14: a 19.6% discount rate against a course
benchmark of 13.8%, with a 50% payment rate against a benchmark of 56%. Discounted
42% harder than its own course and collected worse.

And course 18 appears **twice** in the top five — cohorts 14 and 2. That is the
signal worth acting on. One anomalous cohort is a story; the same course twice is
a pattern in how that course is being sold.

Three honest caveats, all of which belong next to this table if it is ever shown
to anyone:

**The benchmark is the course's own mean**, which the flagged group is part of. A
course with one badly-discounted cohort drags its own benchmark toward it, making
the group look less anomalous than it is.

**A 9-enrollment denominator gives a payment rate that moves 11 points per
student.** The 8+ filter makes the ranking usable, not precise.

**"Over-discounted" is a hypothesis, not a conclusion.** These 21 groups are where
to look, not what is wrong. The model's job is to narrow 383 groups to 21; a human
has to look at the 21.

### Question 10

Finally, ask the model a seventh question: *which payment method do students who pay in full prefer?* Try `fact.groupby("payment_type_key")`. **This is supposed to fail.** Say what the failure means for the model.
> **NOTE:** worksheet 07 question 2 already told you why this column cannot exist on this table.

In [ ]:
print("fact_enrollment columns:")
print("  ", list(fact.columns))
print()
varies = int((tx.groupby("enrl_id").pymt_type_id.nunique() > 1).sum())
print("enrollments paying with MORE THAN ONE payment type:", varies,
      "of", tx.enrl_id.nunique())
print()
print(fact.groupby("payment_type_key").enrollment_count.sum())

```
fact_enrollment columns:
   ['enrollment_id', 'program_key', 'course_key', 'cohort_key', 'student_key',
    'enrollment_date_id', 'promotion_key', 'enrollment_count', 'tuition_amount',
    'discount_amount', 'net_tuition_amount', 'amount_paid_to_date',
    'is_paid_in_full', 'is_cancelled']

enrollments paying with MORE THAN ONE payment type: 1313 of 2243

KeyError: 'payment_type_key'
```

The model cannot answer it, and **the reason is not an oversight.**

**1,313 of 2,243 enrollments — 59% — pay with more than one payment type.** A
student pays the deposit by card and the balance by bank transfer. There is no
single payment method for an enrollment, so `payment_type_key` cannot be a column
on a table whose grain is one row per enrollment. Any value you put there would be
a lie about the other instalments.

Worksheet 07 question 2 found this with a one-line test and it decided the model's
scope. Worksheet 05 question 3 recorded the consequence: `payment_type` is one of
three source tables the model never uses — *out of scope for this fact table*,
which is a different statement from *irrelevant*.

The right answer is the one slide 29 already anticipates in its own notes:

> *For detailed payment analysis, add a separate `fact_payment` table. This would
> turn the model into a galaxy schema.*

`fact_payment` at payment grain, with `payment_type_key`, sharing `dim_course`,
`dim_cohort` and `dim_date` with `fact_enrollment`. Worksheet 04 question 6 built
a minimal version, and question 7 showed the trap that comes with it — a
cross-fact join on `course_id` producing revenue **101x** too large.

**This is what slide 30's validation step is for.** Not to confirm the model
works, but to find its edges. Five of slide 25's six questions are answered
cleanly; a seventh, entirely reasonable, needs a table that does not exist. That
is a scoping decision surfacing — visible now, at design time, rather than in
three months when someone builds the dashboard and finds the column missing.

A model that answers every question you can think of is a model that has not been
questioned hard enough.

**What this sheet established:**

| | |
|---|---|
| row count | 2,400 - 18 = **2,382**, matched — the only check that catches a missing-rows bug |
| nulls | **0 of 14 columns**, earned by `Unknown` members and explicit defaults |
| duplicates | `enrollment_id` clean; **16** on the business grain, and they are *expected* |
| referential integrity | **0 orphans**; 7 on Unknown student, 1,167 on No Promotion |
| business rules | 4 of 4, and **0 of 156** zero-tuition rows falsely marked paid |
| the harness | **7 of 7**, with expectations derived outside the table |
| slide 25 Q1-Q5 | answered — though Q4's unfiltered ranking is five groups of n<10 |
| slide 25 Q6 | **21 of 115** groups discount above and collect below their course benchmark |
| a seventh question | **unanswerable** — 59% of enrollments use multiple payment types |

**The day, in one line each:**

The grain is a claim you test, not a sentence you write. Facts hold measures and
keys; dimensions hold everything you filter by. Additive, semi-additive and
non-additive are properties of a measure *and a grain*. Star by default. The
mapping is an artefact, and half the source will not be in it. Business rules are
decisions, and they belong in a register that computes its own counts. Derived
measures are where the requirements live. Load dimensions first, with `Unknown`
members. Reconcile against something outside the table. Then ask the model the
questions it was built for — and one it was not.